In [6]:
from math import floor

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pulp

# load the data
df = pd.read_csv("Data\\MPAA_model_data.csv")

df['carbon index norm'] = df['carbon index norm'].fillna(df['carbon index norm'].mean())

# List of municipalities IDs
municipalities = df['NM_MUN'].tolist()

# Biodiversity score for each municipality Bio_i
Bio = df.set_index('NM_MUN')['biodiversity_priority_index'].to_dict()

# Extinction risk normalised E_hat_i
E_hat = df.set_index('NM_MUN')['Extinction risk score norm'].to_dict()

# Carbon sequestration potential for each municipality Car_hat_i
Car_hat = df.set_index('NM_MUN')['carbon index norm'].to_dict()

# Land available for planting in each municipality A_i
A = df.set_index('NM_MUN')['eligible_area_ha_mapbiomas'].to_dict()

# Urgency score U_hat_i
U_hat = df.set_index('NM_MUN')['urgency_5yr_index'].to_dict()

# Reversal risk Rev_hat_i
Rev_hat = df.set_index('NM_MUN')['reversal_risk'].to_dict()

# cost per hectare for each municipality c_i
c = df.set_index('NM_MUN')['cost_per_ha'].to_dict()

# max cost per hectare across all municipalities (for cost-effectiveness)
c_max = max(c.values())
c_max

# ========================= Fixed Additional Parameters ======================== 
B = 489850000  # Total budget in dollars
m = 200    # Minimum viable project scale in hectares
K = 560000 # Updated based on new data and assumptions - can implement up to 560,000 hectares per year across all municipalities
N_min = 25       # Minimum number of municipalities to fund (Fairness constraint)
theta = 0.20  # No municipality can receive more than 20% of the total budget B

# weight constraints
alpha_B = 0.5      # Weight for Biodiversity
beta_B = 0.5       # Weight for Extinction Risk

arc_municipalities = [
    "Abaetetuba",
    "Abel Figueiredo",
    "Acará",
    "Ananindeua",
    "Aurora do Pará",
    "Bagre",
    "Baião",
    "Barcarena",
    "Belém",
    "Benevides",
    "Bom Jesus do Tocantins",
    "Brejo Grande do Araguaia",
    "Breu Branco",
    "Bujaru",
    "Cametá",
    "Canaã dos Carajás",
    "Conceição do Araguaia",
    "Concórdia do Pará",
    "Curionópolis",
    "Eldorado do Carajás",
    "Floresta do Araguaia",
    "Goianésia do Pará",
    "Igarapé-Miri",
    "Inhangapi",
    "Irituia",
    "Itupiranga",
    "Jacundá",
    "Limoeiro do Ajuru",
    "Mãe do Rio",
    "Marituba",
    "Mocajuba",
    "Moju",
    "Nova Ipixuna",
    "Oeiras do Pará",
    "Palestina do Pará",
    "Piçarra",
    "Salvaterra",
    "Santa Bárbara do Pará",
    "Santa Izabel do Pará",
    "São Domingos do Araguaia",
    "São Domingos do Capim",
    "São Geraldo do Araguaia",
    "São João do Araguaia",
    "Sapucaia",
    "Tailândia",
    "Tomé-Açu",
    "Tucuruí",
    "Xinguara",
    "Água Azul do Norte",
    "Anajás",
    "Anapu",
    "Bannach",
    "Bonito",
    "Breves",
    "Cachoeira do Arari",
    "Castanhal",
    "Chaves",
    "Colares",
    "Cumaru do Norte",
    "Curralinho",
    "Gurupá",
    "Marabá",
    "Melgaço",
    "Muaná",
    "Novo Repartimento",
    "Ourém",
    "Ourilândia do Norte",
    "Pacajá",
    "Parauapebas",
    "Pau D'Arco",
    "Ponta de Pedras",
    "Portel",
    "Porto de Moz",
    "Redenção",
    "Rio Maria",
    "Santa Cruz do Arari",
    "Santa Luzia do Pará",
    "Santa Maria das Barreiras",
    "Santana do Araguaia",
    "Santo Antônio do Tauá",
    "São Caetano de Odivelas",
    "São Félix do Xingu",
    "São Francisco do Pará",
    "São Miguel do Guamá",
    "São Sebastião da Boa Vista",
    "Senador José Porfírio",
    "Soure",
    "Vigia",
    "Capitão Poço",
    "Dom Eliseu",
    "Garrafão do Norte",
    "Ipixuna do Pará",
    "Nova Esperança do Piriá",
    "Paragominas",
    "Rondon do Pará",
    "Ulianópolis",
    "Viseu"
    ]


In [7]:
dist_df = pd.read_csv("Data\\distance_matrix_named.csv", index_col=0)

# --- 1. SPATIAL PRE-COMPUTATION ---
# Run this once globally so the model creation function is blazing fast.
D_max = 250.0
valid_pairs = []
inv_dist = {}

# Ensure IDs are strings to prevent lookup errors
mun_ids = [str(m) for m in municipalities]
dist_df.index = dist_df.index.astype(str)
dist_df.columns = dist_df.columns.astype(str)

for i in mun_ids:
    for j in mun_ids:
        if i < j: # Ensures we only check each unique pair once
            try:
                dij = float(dist_df.at[i, j])
                if 0 < dij <= D_max:
                    valid_pairs.append((i, j))
                    inv_dist[(i, j)] = 1.0 / dij
            except Exception:
                continue

#print(f"Generated {len(valid_pairs)} valid spatial pairs within {D_max}km threshold.")
# count number of distances in dist_df that are less than D_max
count_within_threshold = 0
for i in mun_ids:
    for j in mun_ids:
        if i < j: # Ensures we only check each unique pair once
            try:
                dij = float(dist_df.at[i, j])
                if 0 < dij <= D_max:
                    count_within_threshold += 1
            except Exception:
                continue
#print(f"Number of municipality pairs within {D_max}km: {count_within_threshold}")

import geopandas as gpd
municipalities_gdf = gpd.read_file("Data\BR_Municipios_2024.shp")

# Filter Pará and project to metric CRS (meters)
para_gdf = municipalities_gdf[municipalities_gdf["NM_UF"] == "Pará"].copy()
    
alloc_df = pd.read_csv("Outputs\\Para Allocations\\Hierarchical_Optimal_Allocation.csv")

alloc_df


,Municipality,Area_ha,Cost
0,Belterra,46989.61,97970000.00
1,Altamira,40589.37,97970000.00
2,Itaituba,42829.47,97970000.00
3,Santarém,41853.16,90298977.07
4,São Félix do Xingu,22968.89,52358803.35
5,Belém,18564.47,38002519.28
6,Anapu,3015.47,6564630.22
7,Afuá,462.92,1000094.85
8,Faro,200.00,553854.00
9,Alenquer,200.00,534338.97


In [8]:
import geopandas as gpd
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import numpy as np
from pathlib import Path

municipalities_gdf = gpd.read_file("Data\\BR_Municipios_2024.shp")


def budget_allocation_para_municipality_graphic(alloc_df, obj_name, all_municipalities):
    allocation = alloc_df.copy()

    allocation = allocation.merge(
        df[['NM_MUN', 'CD_MUN']],
        left_on='Municipality',
        right_on='NM_MUN',
        how='left'
    )

    # list of municipalities in para
    para_municipalities = municipalities_gdf[municipalities_gdf['NM_UF'] == "Pará"]

    para_municipalities['CD_MUN'] = para_municipalities['CD_MUN'].astype(int)

    # merge the geodataframe with allocation datagrame to get the geometry for each municipality
    para_alloc = para_municipalities.merge(
        allocation,
        left_on='CD_MUN',
        right_on='CD_MUN',
        how='left'
    )
    # check how many municipalities have missing allocations (i.e., not selected in the optimal solution)
    print(para_alloc["Cost"].isna().sum(), "municipalities missing allocations")
    print(len(para_alloc), "total municipalities")

    return para_alloc

def plot_budget_allocation_map(para_alloc, obj_name):
    import matplotlib.ticker as mticker

    para_municipalities = municipalities_gdf[municipalities_gdf["NM_UF"] == "Pará"]
    funded = para_alloc[para_alloc["Cost"].notna() & (para_alloc["Cost"] > 0)].copy()

    if funded.empty:
        print("No funded municipalities found (Cost > 0).")
        return

    vmin = funded["Cost"].min()
    vmax = funded["Cost"].max()

    fig = plt.figure(figsize=(12, 12))
    ax_map = fig.add_axes([0.05, 0.08, 0.85, 0.84])

    para_municipalities.plot(
        ax=ax_map, color="#e8e8e8", edgecolor="#aaaaaa", linewidth=0.3
    )

    funded.plot(
        ax=ax_map,
        column="Cost",
        cmap="YlGn",
        edgecolor="#333333",
        linewidth=0.6,
        vmin=vmin,
        vmax=vmax,
        legend=False
    )

    sm = plt.cm.ScalarMappable(cmap="YlGn", norm=plt.Normalize(vmin=vmin, vmax=vmax))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax_map, orientation="vertical", fraction=0.03, pad=0.02)
    cbar.set_label("Reforestation Budget (USD)", fontsize=16)
    cbar.ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
    cbar.ax.tick_params(labelsize=14)

    ax_map.annotate(
        "N", xy=(0.96, 0.95), xycoords="axes fraction",
        ha="center", va="bottom", fontsize=16, fontweight="bold"
    )
    ax_map.annotate(
        "▲", xy=(0.96, 0.92), xycoords="axes fraction",
        ha="center", va="top", fontsize=18
    )

    ax_map.set_title(
        f"Reforestation Budget Allocation Across Pará Municipalities\nObjective: {obj_name}",
        fontsize=18, fontweight="bold", pad=12
    )
    ax_map.set_axis_off()


    # saving the figure to /Figures folder which is a subfolder of the current working directory
    out_dir = Path("Figures/Para Allocation Plots")
    filename = f"Para_Budget_Map_{obj_name.replace(' ', '_')}.png"
    out_path = out_dir / filename
    plt.savefig(out_path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.show()

In [9]:
# --- 2. BASE MODEL CREATION FUNCTION ---
def create_base_model(sense=pulp.LpMaximize):
    model = pulp.LpProblem("PMV_Hierarchical_Model", sense)

    # ==========================================
    # 1. Decision Variables
    # ==========================================
    x = pulp.LpVariable.dicts("x", mun_ids, lowBound=0, cat=pulp.LpContinuous)
    
    y = pulp.LpVariable.dicts("y", mun_ids, cat=pulp.LpBinary)
    
    z = pulp.LpVariable.dicts("z", valid_pairs, lowBound=0, upBound=1, cat=pulp.LpContinuous)

    # ==========================================
    # 2. Core Constraints
    # ==========================================
    # Budget and Capacity Limits
    model += pulp.lpSum(c[i] * x[i] for i in mun_ids) <= B, "Budget_Limit"
    model += pulp.lpSum(x[i] for i in mun_ids) <= K, "Global_Capacity_Limit"
    
    # Geographic Equity Threshold
    model += pulp.lpSum(y[i] for i in mun_ids) >= N_min, "Min_Municipalities"

    # Arc of Deforestation Ring-fenced Minimum
    valid_arc_muns = [i for i in arc_municipalities if str(i) in c]
    model += pulp.lpSum(c[str(i)] * x[str(i)] for i in valid_arc_muns) >= 12350000, "Arc_Minimum"

    # ==========================================
    # 3. Logical Bounds
    # ==========================================
    for i in mun_ids:
        # Cannot restore more than eligible area; forces x=0 if y=0
        model += x[i] <= A[i] * y[i], f"Upper_Bound_Area_{i}"
        
        # Must meet minimum viable scale if selected
        model += x[i] >= m * y[i], f"Min_Viable_Scale_{i}"

        # Geographic equity constraint: no municipality can receive more than theta share of total budget
        model += c[i] * x[i] <= theta * B, f"Max_Budget_Share_{i}"

    # ==========================================
    # 4. Fortet–Glover Linearisation for z_ij
    # ==========================================
    for (i, j) in valid_pairs:
        # These bounds keep the continuous z tight to the binary logic of y_i * y_j
        model += z[(i, j)] <= y[i], f"FG_Upper_i_{i}_{j}"
        model += z[(i, j)] <= y[j], f"FG_Upper_j_{i}_{j}"
        
        # This forces z up to 1 when both y[i] and y[j] are 1
        model += z[(i, j)] >= y[i] + y[j] - 1, f"FG_Lower_{i}_{j}"

    # ==========================================
    # 5. Objective Expressions
    # ==========================================
    obj_bio = pulp.lpSum((alpha_B * Bio[i] + beta_B * E_hat[i]) * x[i] for i in mun_ids)
    obj_urg = pulp.lpSum(U_hat[i] * x[i] for i in mun_ids)
    obj_car = pulp.lpSum(Car_hat[i] * x[i] for i in mun_ids)
    obj_rev = pulp.lpSum(Rev_hat[i] * x[i] for i in mun_ids)
    
    # Spatial penalty is only calculated for pairs within D_max
    obj_spatial = pulp.lpSum(z[(i, j)] * inv_dist[(i, j)] for (i, j) in valid_pairs)

    return model, x, y, obj_bio, obj_urg, obj_car, obj_rev, obj_spatial


In [10]:
tolerances = pd.read_csv("Outputs\\lexicographic_tolerances.csv")
tolerances

,Objective,Nadir,Utopia,range,tolerance
0,Bio_and_extinction,1036.651394,145998.502805,144961.851411,1449.618514
1,Carbon,5313.825487,193410.001968,188096.176481,1880.961765
2,Urgency,38.212297,141990.933009,141952.720712,1419.527207
3,Reversal Risk,216855.272249,0.000000,216855.272249,2168.552722
4,Spatial Penalty,3.457952,0.086386,3.371566,0.033716


In [11]:
import pandas as pd
import pulp


# Extract the absolute ranges for each objective
range_bio = tolerances.loc[tolerances['Objective'] == 'Bio_and_extinction', 'range'].values[0]
range_car = tolerances.loc[tolerances['Objective'] == 'Carbon', 'range'].values[0]
range_urg = tolerances.loc[tolerances['Objective'] == 'Urgency', 'range'].values[0]
range_rev = tolerances.loc[tolerances['Objective'] == 'Reversal Risk', 'range'].values[0]

# Define the alpha values we want to test
alpha_values = [0.05, 0.10, 0.15]

# Dictionary to store the summary of actual values for comparison
sensitivity_results = []

for alpha in alpha_values:
    print(f"\n{'='*70}")
    print(f"RUNNING LEXICOGRAPHIC SENSITIVITY ANALYSIS FOR ALPHA = {alpha} ({alpha*100}%)")
    print(f"{'='*70}")
    
    # Calculate exact mathematical bounds/tolerances for this run
    tol_bio = alpha * range_bio
    tol_car = alpha * range_car
    tol_urg = alpha * range_urg
    tol_rev = alpha * range_rev

    # ==========================================
    # STEP 1: Maximize Biodiversity & Threat (F1)
    # ==========================================
    print("\nSolving Step 1: Biodiversity and Threat...")
    model, x, y, obj_bio, obj_urg, obj_car, obj_rev, obj_spatial = create_base_model(pulp.LpMaximize)
    model += obj_bio, "Objective_F1"

    model.solve(pulp.GUROBI(msg=False))
    if pulp.LpStatus[model.status] != "Optimal":
        raise RuntimeError(f"Step 1 failed for alpha {alpha}")

    z1_star = pulp.value(model.objective)
    print(f"Theoretical Max F1: {z1_star:.2f} | Allowed degradation limit: >= {z1_star - tol_bio:.2f}")

    # ==========================================
    # STEP 2: Maximize Carbon Sequestration (F2)
    # ==========================================
    print("\nSolving Step 2: Carbon Sequestration...")
    model, x, y, obj_bio, obj_urg, obj_car, obj_rev, obj_spatial = create_base_model(pulp.LpMaximize)
    
    # F1 is MAX, so we subtract tolerance to allow it to drop slightly
    model += obj_bio >= (z1_star - tol_bio), "Preserve_F1_Biodiversity"
    model += obj_car, "Objective_F2"

    model.solve(pulp.GUROBI(msg=False))
    if pulp.LpStatus[model.status] != "Optimal":
        raise RuntimeError(f"Step 2 failed for alpha {alpha}")

    z2_star = pulp.value(model.objective)
    print(f"Theoretical Max F2: {z2_star:.2f} | Allowed degradation limit: >= {z2_star - tol_car:.2f}")

    # ==========================================
    # STEP 3: Maximize Urgency (F3)
    # ==========================================
    print("\nSolving Step 3: Maximize Urgency...")
    model, x, y, obj_bio, obj_urg, obj_car, obj_rev, obj_spatial = create_base_model(pulp.LpMaximize)
    
    model += obj_bio >= (z1_star - tol_bio), "Preserve_F1_Biodiversity"
    model += obj_car >= (z2_star - tol_car), "Preserve_F2_Carbon"
    model += obj_urg, "Objective_F3"

    model.solve(pulp.GUROBI(msg=False))
    if pulp.LpStatus[model.status] != "Optimal":
        raise RuntimeError(f"Step 3 failed for alpha {alpha}")

    z3_star = pulp.value(model.objective)
    print(f"Theoretical Max F3: {z3_star:.2f} | Allowed degradation limit: >= {z3_star - tol_urg:.2f}")

    # ==========================================
    # STEP 4: Minimize Reversal Risk (F4)
    # ==========================================
    print("\nSolving Step 4: Minimizing Reversal Risk...")
    model, x, y, obj_bio, obj_urg, obj_car, obj_rev, obj_spatial = create_base_model(pulp.LpMinimize)
    
    model += obj_bio >= (z1_star - tol_bio), "Preserve_F1_Biodiversity"
    model += obj_car >= (z2_star - tol_car), "Preserve_F2_Carbon"
    model += obj_urg >= (z3_star - tol_urg), "Preserve_F3_Urgency"
    model += obj_rev, "Objective_F4"

    model.solve(pulp.GUROBI(msg=False))
    if pulp.LpStatus[model.status] != "Optimal":
        raise RuntimeError(f"Step 4 failed for alpha {alpha}")

    z4_star = pulp.value(model.objective)
    # F4 is MIN, so we ADD tolerance to allow it to get slightly worse (higher)
    print(f"Theoretical Min F4: {z4_star:.2f} | Allowed degradation limit: <= {z4_star + tol_rev:.2f}")

    # ==========================================
    # STEP 5: Minimize Spatial Connectivity/Dispersion (F5)
    # ==========================================
    print("\nSolving Step 5: Minimizing Spatial Dispersion...")
    model, x, y, obj_bio, obj_urg, obj_car, obj_rev, obj_spatial = create_base_model(pulp.LpMinimize)
    
    model += obj_bio >= (z1_star - tol_bio), "Preserve_F1_Biodiversity"
    model += obj_car >= (z2_star - tol_car), "Preserve_F2_Carbon"
    model += obj_urg >= (z3_star - tol_urg), "Preserve_F3_Urgency"
    model += obj_rev <= (z4_star + tol_rev), "Preserve_F4_Reversal"
    model += obj_spatial, "Objective_F5"

    model.solve(pulp.GUROBI(msg=False)) # Set msg=True if you need to debug gaps
    if pulp.LpStatus[model.status] != "Optimal":
        raise RuntimeError(f"Step 5 failed for alpha {alpha}")

    # Record actual objective values realized at the final step
    actual_f1 = pulp.value(obj_bio)
    actual_f2 = pulp.value(obj_car)
    actual_f3 = pulp.value(obj_urg)
    actual_f4 = pulp.value(obj_rev)
    actual_f5 = pulp.value(obj_spatial)
    
    print(f"Actual Realized F5: {actual_f5:.4f}")
    
    # Save the actualized results to our summary
    sensitivity_results.append({
        'Alpha': alpha,
        'F1_Bio (Max)': actual_f1,
        'F2_Car (Max)': actual_f2,
        'F3_Urg (Max)': actual_f3,
        'F4_Rev (Min)': actual_f4,
        'F5_Spa (Min)': actual_f5
    })

# ==========================================
# FINAL SENSITIVITY SUMMARY OUTPUT
# ==========================================
print("\n" + "="*80)
print("SENSITIVITY ANALYSIS RESULTS: COMPARING ACTUAL OBJECTIVE ATTAINMENT")
print("="*80)
results_df = pd.DataFrame(sensitivity_results).set_index('Alpha')
print(results_df.to_markdown())


RUNNING LEXICOGRAPHIC SENSITIVITY ANALYSIS FOR ALPHA = 0.05 (5.0%)

Solving Step 1: Biodiversity and Threat...
Theoretical Max F1: 145998.50 | Allowed degradation limit: >= 138750.41

Solving Step 2: Carbon Sequestration...
Theoretical Max F2: 137265.55 | Allowed degradation limit: >= 127860.74

Solving Step 3: Maximize Urgency...
Theoretical Max F3: 97654.43 | Allowed degradation limit: >= 90556.79

Solving Step 4: Minimizing Reversal Risk...
Theoretical Min F4: 116240.16 | Allowed degradation limit: <= 127082.92

Solving Step 5: Minimizing Spatial Dispersion...
Actual Realized F5: 0.1083

RUNNING LEXICOGRAPHIC SENSITIVITY ANALYSIS FOR ALPHA = 0.1 (10.0%)

Solving Step 1: Biodiversity and Threat...
Theoretical Max F1: 145998.50 | Allowed degradation limit: >= 131502.32

Solving Step 2: Carbon Sequestration...
Theoretical Max F2: 147203.77 | Allowed degradation limit: >= 128394.15

Solving Step 3: Maximize Urgency...
Theoretical Max F3: 112037.60 | Allowed degradation limit: >= 97842.

From the results, $\alpha=0.05$ is chosen for the rest of the models